# Study 891 — Insurance Float Engine — the teardown

The excess-vs-excess Sharpe race, the HAC *t* on the return difference, the bootstrap Sharpe CIs, the CAPM and the decisive two-factor decomposition, the era cut, the costed rotation/isolation trades, and the planted-edge synthetic control.

In [1]:
R = {'start': '2007-06-30', 'end': '2026-06-30', 'n': 229, 'fp': '3a54dbc09ab6', 'kie_cagr': 7.86, 'kie_vol': 21.73, 'kie_sharpe': 0.398, 'kie_dd': -69.7, 'iak_cagr': 6.8, 'iak_vol': 20.85, 'iak_sharpe': 0.359, 'iak_dd': -72.2, 'spy_cagr': 10.68, 'spy_vol': 15.58, 'spy_sharpe': 0.644, 'spy_dd': -50.8, 'kbe_cagr': 3.21, 'kbe_vol': 27.64, 'kbe_sharpe': 0.209, 'kbe_dd': -76.6, 'kie_adv': -0.246, 'kie_diff': -1.39, 'kie_tdiff': -0.49, 'iak_adv': -0.285, 'iak_diff': -2.56, 'iak_tdiff': -0.9, 'kie_ci': (-0.06, 0.966), 'iak_ci': (-0.116, 0.949), 'spy_ci': (0.18, 1.182), 'kie_capm_a': -2.48, 'kie_capm_ta': -0.84, 'kie_capm_b': 1.109, 'iak_capm_a': -3.1, 'iak_capm_ta': -0.95, 'iak_capm_b': 1.053, 'kie_two_a': -0.11, 'kie_two_ta': -0.04, 'kie_two_load': 0.357, 'kie_two_tload': 6.27, 'iak_two_a': -0.96, 'iak_two_ta': -0.34, 'iak_two_load': 0.322, 'iak_two_tload': 5.81, 'kie_kbe': 2.89, 'kie_kbe_t': 0.96, 'iak_kbe': 1.71, 'iak_kbe_t': 0.58, 'era_gfc': 0.118, 'era_1015': -0.125, 'era_1620': -0.374, 'era_2126': -0.157, 'era_post': -0.207, 'rot_net': 9.51, 'rot_sharpe': 0.49, 'rot_mkt': 11.41, 'rot_ins': 10.03, 'rot_switches': 30, 'iso_gross': -1.39, 'iso_tg': -0.49, 'iso_net': -1.89, 'iso_tn': -0.66, 'iso_charge': 0.5, 'syn_null_adv': 0.092, 'syn_null_ta': 1.48, 'syn_edge_adv': 0.305, 'syn_edge_ta': 4.34, 'syn_load': 0.387, 'syn_tload': 10.7}

## The race — excess Sharpe (both legs minus BIL) + HAC *t* on the diff

In [2]:
print(f"n = {R['n']} months  {R['start']} -> {R['end']}  fp {R['fp']}")
print(f"KIE exSharpe {R['kie_sharpe']:+.3f}  vs SPY {R['spy_sharpe']:+.3f}  "
      f"advantage {R['kie_adv']:+.3f}  | KIE-SPY {R['kie_diff']:+.2f}%/yr "
      f"(HAC t={R['kie_tdiff']:+.2f})")
print(f"IAK exSharpe {R['iak_sharpe']:+.3f}  vs SPY {R['spy_sharpe']:+.3f}  "
      f"advantage {R['iak_adv']:+.3f}  | IAK-SPY {R['iak_diff']:+.2f}%/yr "
      f"(HAC t={R['iak_tdiff']:+.2f})")

n = 229 months  2007-06-30 -> 2026-06-30  fp 3a54dbc09ab6
KIE exSharpe +0.398  vs SPY +0.644  advantage -0.246  | KIE-SPY -1.39%/yr (HAC t=-0.49)
IAK exSharpe +0.359  vs SPY +0.644  advantage -0.285  | IAK-SPY -2.56%/yr (HAC t=-0.90)


## Bootstrap 95% CI on the excess Sharpe — you can't separate insurer from zero

Circular block bootstrap. SPY's interval clears zero; the insurer intervals include it.

In [3]:
for t in ['kie','iak','spy']:
    lo,hi = R[t+'_ci']
    print(f"{t.upper()}: exSharpe {R[t+'_sharpe']:+.3f}  CI[{lo:+.3f}, {hi:+.3f}]")

KIE: exSharpe +0.398  CI[-0.060, +0.966]
IAK: exSharpe +0.359  CI[-0.116, +0.949]
SPY: exSharpe +0.644  CI[+0.180, +1.182]


## CAPM — insurer excess on market excess (no alpha over the market)

In [4]:
print(f"KIE: alpha {R['kie_capm_a']:+.2f}%/yr (t={R['kie_capm_ta']:+.2f})  beta {R['kie_capm_b']:.3f}")
print(f"IAK: alpha {R['iak_capm_a']:+.2f}%/yr (t={R['iak_capm_ta']:+.2f})  beta {R['iak_capm_b']:.3f}")

KIE: alpha -2.48%/yr (t=-0.84)  beta 1.109
IAK: alpha -3.10%/yr (t=-0.95)  beta 1.053


## The decisive test — add the bank-sector factor, and the alpha dies

`insurer_ex = alpha + beta*market_ex + s*(bank_ex - market_ex)`. A float premium would survive; sector beta is absorbed.

In [5]:
print(f"KIE: alpha {R['kie_two_a']:+.2f}%/yr (t={R['kie_two_ta']:+.2f})  "
      f"bank load {R['kie_two_load']:+.3f} (t={R['kie_two_tload']:+.2f})")
print(f"IAK: alpha {R['iak_two_a']:+.2f}%/yr (t={R['iak_two_ta']:+.2f})  "
      f"bank load {R['iak_two_load']:+.3f} (t={R['iak_two_tload']:+.2f})")
print()
print('The insurer alpha over [market + bank factor] is indistinguishable from zero.')

KIE: alpha -0.11%/yr (t=-0.04)  bank load +0.357 (t=+6.27)
IAK: alpha -0.96%/yr (t=-0.34)  bank load +0.322 (t=+5.81)

The insurer alpha over [market + bank factor] is indistinguishable from zero.


## Within-financials — insurers DID beat banks (a different, weaker claim)

In [6]:
print(f"KIE - KBE: {R['kie_kbe']:+.2f}%/yr (HAC t={R['kie_kbe_t']:+.2f})")
print(f"IAK - KBE: {R['iak_kbe']:+.2f}%/yr (HAC t={R['iak_kbe_t']:+.2f})")
print('Real direction (float < spread-leverage in risk), but not t>=2 on 19 years.')

KIE - KBE: +2.89%/yr (HAC t=+0.96)
IAK - KBE: +1.71%/yr (HAC t=+0.58)
Real direction (float < spread-leverage in risk), but not t>=2 on 19 years.


## Era cut — the deficit isn't one crisis; it's every calm era

KIE minus SPY excess-Sharpe advantage by era. Positive only in the 2007-09 crash.

In [7]:
for tag,v in [('GFC 2007-09',R['era_gfc']),('2010-15',R['era_1015']),
              ('2016-20',R['era_1620']),('2021-26',R['era_2126']),
              ('post-GFC 2010+',R['era_post'])]:
    flag = '  <- only positive era (fell less in the crash)' if v>0 else ''
    print(f"{tag:16s} KIE-SPY advantage {v:+.3f}{flag}")

GFC 2007-09      KIE-SPY advantage +0.118  <- only positive era (fell less in the crash)
2010-15          KIE-SPY advantage -0.125
2016-20          KIE-SPY advantage -0.374
2021-26          KIE-SPY advantage -0.157
post-GFC 2010+   KIE-SPY advantage -0.207


## Tradability — nothing to pocket over the market

In [8]:
print(f"(a) 1-month-lag rotation (own KIE when it's led SPY 12m): net {R['rot_net']:+.2f}%/yr")
print(f"    vs always-SPY {R['rot_mkt']:+.2f}%/yr  always-KIE {R['rot_ins']:+.2f}%/yr  "
      f"({R['rot_switches']} switches) -> underperforms just owning the market")
print(f"(b) isolation (long KIE / short SPY, borrow+costs {R['iso_charge']:.2f}%/yr): "
      f"gross {R['iso_gross']:+.2f}%/yr (t={R['iso_tg']:+.2f}) -> "
      f"net {R['iso_net']:+.2f}%/yr (t={R['iso_tn']:+.2f})")

(a) 1-month-lag rotation (own KIE when it's led SPY 12m): net +9.51%/yr
    vs always-SPY +11.41%/yr  always-KIE +10.03%/yr  (30 switches) -> underperforms just owning the market
(b) isolation (long KIE / short SPY, borrow+costs 0.50%/yr): gross -1.39%/yr (t=-0.49) -> net -1.89%/yr (t=-0.66)


## Synthetic positive control — the machinery is unbiased

Live: the detector must NOT fire on the null and must recover a planted +4 %/yr edge.

In [9]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from insurance_float import data, strategy as st
null = st.synthetic_detect(data.synthetic_world(edge_ann=0.0, seed=891))
edge = st.synthetic_detect(data.synthetic_world(edge_ann=0.04, seed=891))
print(f"null  : Sharpe adv {null['advantage']:+.3f}, CAPM alpha t={null['capm_t_alpha']:+.2f}, "
      f"two-factor alpha t={null['two_t_alpha']:+.2f}  (all quiet)")
print(f"planted: Sharpe adv {edge['advantage']:+.3f}, CAPM alpha t={edge['capm_t_alpha']:+.2f}, "
      f"two-factor alpha t={edge['two_t_alpha']:+.2f}  (recovered, survives bank control)")
print(f"bank loading always found: {edge['load_bank']:+.3f} (t={edge['t_load_bank']:+.2f})")

null  : Sharpe adv +0.092, CAPM alpha t=+1.48, two-factor alpha t=+1.40  (all quiet)
planted: Sharpe adv +0.305, CAPM alpha t=+4.34, two-factor alpha t=+4.40  (recovered, survives bank control)
bank loading always found: +0.387 (t=+10.70)


## Verdict

- **Signal — None.** No edge over the market attributable to float — the sign runs the wrong way. KIE/IAK excess Sharpe 0.40/0.36 vs SPY 0.64; advantage -0.25/-0.28 (HAC *t* on the diff -0.49/-0.90); CAPM alpha -2.5/-3.1 %/yr; and the two-factor alpha collapses to -0.11 %/yr (*t* = -0.04) against a financial-sector loading of +0.36 (*t* = 6.3). Negative in every calm era; bootstrap can't clear zero. Short 19-year sample, one crisis inside it. (Insurers > banks by +2.9 %/yr at *t* = 0.96 — a different, unclean claim.)
- **Tradability — Mirage.** The isolation trade nets -1.89 %/yr; the rotation nets +9.5 %/yr below always-SPY's 11.4 %/yr. The engine is sector beta — cheaper, and shallower-drawdown, as plain SPY.